In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow
from tensorflow import keras
from keras.layers import Embedding,Dense,LSTM
from keras.models import Sequential



In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "shortjokes.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "abhinavmoudgil95/short-jokes",
    file_path,
)

print("First 5 records:", df.head())


/tmp/ipykernel_3136/1650386988.py:6: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 9.82M/9.82M [00:00<00:00, 26.1MB/s]

Extracting zip of shortjokes.csv...


First 5 records:    ID                                               Joke
0   1  [me narrating a documentary about narrators] "...
1   2  Telling my daughter garlic is good for you. Go...
2   3  I've been going through a really rough period ...
3   4  If I could have dinner with anyone, dead or al...
4   5     Two guys walk into a bar. The third guy ducks.


In [3]:
df.shape

(231657, 2)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 231657 entries, 0 to 231656
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   ID      231657 non-null  int64 
 1   Joke    231657 non-null  object
dtypes: int64(1), object(1)
memory usage: 3.5+ MB


In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=10000)
df = df.sample(15000)

In [6]:
df.isnull().sum()

,0
ID,0
Joke,0


In [7]:
seq = tokenizer.fit_on_texts(df['Joke'])

In [8]:
len(tokenizer.word_index)

18817

In [9]:
input_sequences = []

for sentence in df['Joke']:
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

    if len(tokenized_sentence) > 1:
        for i in range(1, len(tokenized_sentence)):
            input_sequences.append(tokenized_sentence[:i+1])

In [10]:
input_sequences

[[2567, 43],
 [2567, 43, 4],
 [2567, 43, 4, 6933],
 [2567, 43, 4, 6933, 187],
 [2567, 43, 4, 6933, 187, 404],
 [2567, 43, 4, 6933, 187, 404, 2],
 [2567, 43, 4, 6933, 187, 404, 2, 268],
 [2567, 43, 4, 6933, 187, 404, 2, 268, 3098],
 [2567, 43, 4, 6933, 187, 404, 2, 268, 3098, 105],
 [2567, 43, 4, 6933, 187, 404, 2, 268, 3098, 105, 1904],
 [2567, 43, 4, 6933, 187, 404, 2, 268, 3098, 105, 1904, 2822],
 [2567, 43, 4, 6933, 187, 404, 2, 268, 3098, 105, 1904, 2822, 2],
 [2567, 43, 4, 6933, 187, 404, 2, 268, 3098, 105, 1904, 2822, 2, 318],
 [2567, 43, 4, 6933, 187, 404, 2, 268, 3098, 105, 1904, 2822, 2, 318, 99],
 [2567,
  43,
  4,
  6933,
  187,
  404,
  2,
  268,
  3098,
  105,
  1904,
  2822,
  2,
  318,
  99,
  5476],
 [2567,
  43,
  4,
  6933,
  187,
  404,
  2,
  268,
  3098,
  105,
  1904,
  2822,
  2,
  318,
  99,
  5476,
  9672],
 [2567,
  43,
  4,
  6933,
  187,
  404,
  2,
  268,
  3098,
  105,
  1904,
  2822,
  2,
  318,
  99,
  5476,
  9672,
  1321],
 [2567,
  43,
  4,
  6933,
  

In [11]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

max_len = max(len(seq) for seq in input_sequences)

input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

In [12]:
input_sequences

array([[   0,    0,    0, ...,    0, 2567,   43],
       [   0,    0,    0, ..., 2567,   43,    4],
       [   0,    0,    0, ...,   43,    4, 6933],
       ...,
       [   0,    0,    0, ...,   11,   33, 1333],
       [   0,    0,    0, ...,   33, 1333,    4],
       [   0,    0,    0, ..., 1333,    4,  178]], dtype=int32)

In [13]:
x=input_sequences[:,:-1]
y=input_sequences[:,-1]

In [14]:
x

array([[   0,    0,    0, ...,    0,    0, 2567],
       [   0,    0,    0, ...,    0, 2567,   43],
       [   0,    0,    0, ..., 2567,   43,    4],
       ...,
       [   0,    0,    0, ..., 5400,   11,   33],
       [   0,    0,    0, ...,   11,   33, 1333],
       [   0,    0,    0, ...,   33, 1333,    4]], dtype=int32)

In [15]:
y.shape

(241156,)

In [28]:
model = Sequential([
    Embedding(10000, 64, input_length=max_len-1),
    LSTM(150, return_sequences=True),
    LSTM(96,return_sequences=True),
    LSTM(64),
    Dense(10000, activation='softmax')
])
model.build((None, max_len - 1))
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 44, 64)         │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 44, 150)        │       129,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 44, 96)         │        94,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10000)          │       650,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,555,064 (5.93 MB)

 Trainable params: 1,555,064 (5.93 MB)

 Non-trainable params: 0 (0.00 B)

In [29]:

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [30]:
model.summary()


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 44, 64)         │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 44, 150)        │       129,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 44, 96)         │        94,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10000)          │       650,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,555,064 (5.93 MB)

 Trainable params: 1,555,064 (5.93 MB)

 Non-trainable params: 0 (0.00 B)

In [31]:
model.fit(x, y, epochs=1)

7537/7537 ━━━━━━━━━━━━━━━━━━━━ 87s 11ms/step - accuracy: 0.0592 - loss: 6.6709


In [32]:
history=model.fit(x, y, epochs=30, validation_split=0.2)

Epoch 1/30
6029/6029 ━━━━━━━━━━━━━━━━━━━━ 77s 13ms/step - accuracy: 0.1019 - loss: 6.0756 - val_accuracy: 0.1089 - val_loss: 5.9717
Epoch 2/30
6029/6029 ━━━━━━━━━━━━━━━━━━━━ 75s 13ms/step - accuracy: 0.1208 - loss: 5.7777 - val_accuracy: 0.1219 - val_loss: 5.9600
Epoch 3/30
6029/6029 ━━━━━━━━━━━━━━━━━━━━ 75s 12ms/step - accuracy: 0.1349 - loss: 5.5687 - val_accuracy: 0.1330 - val_loss: 5.9923
Epoch 4/30
6029/6029 ━━━━━━━━━━━━━━━━━━━━ 75s 13ms/step - accuracy: 0.1453 - loss: 5.4025 - val_accuracy: 0.1395 - val_loss: 6.0092
Epoch 5/30
6029/6029 ━━━━━━━━━━━━━━━━━━━━ 75s 12ms/step - accuracy: 0.1530 - loss: 5.2640 - val_accuracy: 0.1447 - val_loss: 6.0613
Epoch 6/30
6029/6029 ━━━━━━━━━━━━━━━━━━━━ 76s 13ms/step - accuracy: 0.1588 - loss: 5.1463 - val_accuracy: 0.1461 - val_loss: 6.1096
Epoch 7/30
6029/6029 ━━━━━━━━━━━━━━━━━━━━ 76s 13ms/step - accuracy: 0.1633 - loss: 5.0404 - val_accuracy: 0.1495 - val_loss: 6.2040
Epoch 8/30
6029/6029 ━━━━━━━━━━━━━━━━━━━━ 75s 13ms/step - accuracy: 0.1688 -

In [34]:
import numpy as np

def generate_text(seed_text, next_words=10):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')

        predicted = np.argmax(model.predict(token_list), axis=-1)

        for word, index in tokenizer.word_index.items():
            if index == predicted:
                seed_text += " " + word
                break

    return seed_text

In [35]:
print(generate_text("why did the", 10))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 274ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
why did the chicken cross the road to get to the other side


In [36]:
print(generate_text("harm", 10))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
harm to to tell me i don't know what i was


In [37]:
print(generate_text("happy", 20))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
happy joke is the most popular girl i was like tough syndrome i guess you could say i can do it


In [29]:
print(generate_text("road", 20))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
road rage and population in my jeans and use a mass alarm laugh where my best friends say when you want


In [32]:
print(generate_text("chicken", 20))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
chicken did you hear about the guy who got up in the hospital he was charged with coming and coming confused


In [39]:
print(generate_text("yash", 20))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
yash old macdonald had to remove the wheel on the first floor and r to the other side of the world
